# GT fidelity -- original page vs. generated vector ground truth

How faithfully does the raster-to-vector ground truth reproduce the page it was taken from? For each page
this renders the **original** page and the **ground truth** at the same dpi and scores them with the
benchmark's own SSIM / PSNR (`vector_metrics.image_similarity` on `vector_render.render_gray` renders).

Two ground truths are compared:

* **visible GT** -- what `raster_label.raster_geometry_for_page` produces now
  (`visible_geometry.visible_geometry_for_page`): paper-coloured paint dropped, clipped / dashed /
  covered pieces cut to the runs the render actually shows. This is the GT the benchmark
  (`vector_to_raster`) and `DeepVectoriser.prep_dataset` use.
* **all-vectors GT** -- the old paint-blind GT (every piece of every path), for comparison.

Logic: `raster_parser/notebooks/gt_fidelity_helpers.py`. Output: `outputs/gt_fidelity_showcase/<timestamp>/`
(`summary.csv`, `pages/<stem>_p<N>.png`, `cache/` of GT renders).

### Why the scores can't reach 1

* GT is **centre-lines at stroke width**: a filled area in the original is only its outline in the GT,
  and line caps / joins differ slightly.
* **Font text and embedded images** have no vector paths, so no GT. `MASK_NATIVE_TEXT` / `MASK_IMAGES`
  paint their boxes white in every image before scoring (on by default).
* Shadings and annotations are rendered in the original but have no GT either.

Read the numbers *relatively*: visible GT should score at least as well as all-vectors GT, and the
diff panel shows where they differ (red = ink with no GT, blue = GT over paper).

## 0 - Configuration

In [ ]:
PDF_PATHS = [
    "../references/test_pdfs_1.pdf",
]
PAGES = None  # list of page indices applied to every PDF, or None = every page
MAX_PAGES_PER_PDF = 3  # None = no cap
DPI = 150  # comparison resolution (the benchmark's VectorMetricConfig.render_dpi)
MASK_NATIVE_TEXT = True
MASK_IMAGES = True
SHOW_PAGES = True  # draw one figure row per page inline (always saved to disk)

## 1 - Bootstrap

In [ ]:
import os, sys

_root = os.path.abspath(os.path.join(".."))
if _root not in sys.path:
    sys.path.insert(0, _root)

from IPython.display import Image, Markdown, display
import matplotlib.pyplot as plt

from notebooks._helpers import gt_fidelity_helpers as h

run_dir = h.new_run_dir()
cache_dir = run_dir / "cache"
print("output:", run_dir)

## 2 - Score every page

Per page: original render | visible GT | all-vectors GT | diff of the visible GT. Building the GT renders
the page at 300 dpi and runs the colour split (a few seconds per large sheet, ~1.3 GB peak for A1);
renders are cached in `cache/`.

In [ ]:
results = []
for pdf in PDF_PATHS:
    for page in h.page_indices(pdf, PAGES, MAX_PAGES_PER_PDF):
        r = h.evaluate_page(
            pdf,
            page,
            dpi=DPI,
            mask_text=MASK_NATIVE_TEXT,
            mask_images=MASK_IMAGES,
            cache_dir=cache_dir,
        )
        results.append(r)
        fig = h.page_figure(r)
        png = h.save_page(r, run_dir, fig)
        plt.close(fig)
        print(
            f"{r.label}: visible SSIM {r.ssim_visible:.3f} / PSNR {r.psnr_visible:.1f} dB | "
            f"all SSIM {r.ssim_all:.3f} / PSNR {r.psnr_all:.1f} dB  ({r.seconds:.1f}s)"
        )
        if SHOW_PAGES:
            display(Image(filename=str(png)))

## 3 - Summary

`vectors` / `paint_dropped`: drawing paths on the page and how many were dropped as paper-coloured;
`pieces_all`: pieces in the all-vectors GT; `pieces_invisible` / `pieces_cut`: pieces with no visible run /
cut to part of their length; `short_runs`: partial runs under 3 px dropped; `gt_pieces_visible`: pieces in
the visible GT. Mean rows per PDF (and overall) at the bottom.

In [ ]:
rows = h.summary_rows(results)
display(Markdown(h.markdown_table(rows)))
print("saved", h.save_summary(rows, run_dir))